# E-to-E minimum-weight method comparison

This notebook is a minimum-weight companion to `ee_backbone_method_comparison.ipynb`. It uses the same excitatory-only matrix, `matrices/mij_EE_matrix.csv`, and keeps the original notebook's structure: prepare the graph, run method families, combine results, write CSV artifacts, and print comparison tables at the bottom.

A literal minimum-weight simple path is degenerate: every seed could choose the one-node path and score zero. To make the adaptations useful, this notebook uses a **coverage-first minimum** rule for variable-length path methods: first visit as many nodes as the method can support, then minimize summed positive transition weight among paths with that coverage. Hamiltonian methods already force all 32 excitatory nodes, so they are ordinary minimum-weight Hamiltonian paths.

In [1]:
from pathlib import Path
import math
import time

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

import numpy as np
import pandas as pd

from ee_backbone_analysis import (
    DSU,
    EPS,
    build_tree_adjacency,
    describe_backbone_path,
    edges_from_path,
    path_to_records,
    prepare_excitatory_matrix_data,
    total_score,
    write_setup_outputs,
)
from ee_backbone_ordering import (
    feedback_arc_ordering,
    ordering_edge_table,
    ordering_metrics,
)

INPUT_CSV = Path("matrices/mij_EE_matrix.csv")
OUTDIR = Path("ee_backbone_min_weight_comparison_outputs")

BB_TIME_LIMIT_PER_SEED = 0.25
BB_NODE_LIMIT_PER_SEED = 50_000
NOTEBOOK_EXACT_DP_MAX_E_NODES = 22
BEAM_WIDTH = 5_000

MILP_TIME_LIMIT_PER_SEED = 30
HAMILTONIAN_TIME_LIMIT_PER_SEED = 30

ORDERING_RESTARTS = 50
ORDERING_NULL_DRAWS = 2_000
CANONICAL_EC_START = "EC LI II Pyramidal Fan"


## Prepare data

The graph preparation mirrors the original EE-only notebook. The matrix is clipped to positive E-to-E transitions, diagonal transitions are excluded, and every row/column label is treated as an excitatory node.

In [2]:
data = prepare_excitatory_matrix_data(INPUT_CSV, eps=EPS)
write_setup_outputs(data, OUTDIR)

print(f"E-to-E matrix: {data.full_matrix.shape[0]} x {data.full_matrix.shape[1]}")
print(f"Excitatory nodes retained: {len(data.excitatory_nodes)}")
print(f"Positive off-diagonal candidate transitions: {(data.transition_weights > EPS).sum()}")
print(f"Canonical EC start present: {CANONICAL_EC_START in data.excitatory_nodes}")


E-to-E matrix: 32 x 32
Excitatory nodes retained: 32
Positive off-diagonal candidate transitions: 355
Canonical EC start present: True


## Minimum-weight helper functions

These helpers adapt the original method families to minimum-weight analysis. For variable-length paths, comparisons are lexicographic: higher coverage first, lower weight second. This prevents the trivial zero-edge answer from dominating every method.

In [3]:
def _path_score(path):
    return total_score(path, data.transition_weights, data.self_weights)


def _path_is_valid(path):
    return all(data.transition_weights[u, v] > EPS for u, v in zip(path[:-1], path[1:]))


def _coverage_min_better(path, score, best_path, best_score):
    if len(path) > len(best_path):
        return True
    if len(path) == len(best_path) and score < best_score - EPS:
        return True
    return False


def _method_record(seed, method, path, extra=None):
    return path_to_records(seed, method, path, data, extra or {})


def _method_edges(seed, method, path):
    return edges_from_path(seed, method, path, data)


## Greedy Tree/Path

The minimum greedy path starts at each seed and repeatedly takes the smallest positive outgoing transition to an unvisited excitatory node. It keeps walking until the current node has no positive transition to an unvisited node.

In [4]:
def minimum_greedy_path(seed, data):
    visited = {seed}
    path = [seed]
    current = seed

    while True:
        candidates = [(j, w) for j, w in data.adjacency[current] if j not in visited]
        if not candidates:
            break
        next_node, _ = min(candidates, key=lambda item: (item[1], data.excitatory_nodes[item[0]]))
        visited.add(next_node)
        path.append(next_node)
        current = next_node
    return path


greedy_results = []
greedy_edges = []
for seed in range(len(data.excitatory_nodes)):
    path = minimum_greedy_path(seed, data)
    method = "greedy_tree_path"
    greedy_results.append(_method_record(seed, method, path, {"objective_direction": "coverage-first minimum"}))
    greedy_edges.extend(_method_edges(seed, method, path))

greedy_df = pd.DataFrame(greedy_results)
greedy_df.head()


,method,seed,length_edges,n_nodes_visited,transition_weight_sum,summed_weight,terminus,path,objective_direction
0,greedy_tree_path,CA1 Pyramidal,8,9,2020.848183,2020.848183,CA1 Radiatum Giant,CA1 Pyramidal -> MEC LV Superficial Pyramidal ...,coverage-first minimum
1,greedy_tree_path,CA1 Radiatum Giant,8,9,2065.086525,2065.086525,SUB CA1 Projecting Pyramidal,CA1 Radiatum Giant -> CA1 Pyramidal -> MEC LV ...,coverage-first minimum
2,greedy_tree_path,CA2 Pyramidal,9,10,2328.151031,2328.151031,SUB CA1 Projecting Pyramidal,CA2 Pyramidal -> CA1 Radiatum Giant -> CA1 Pyr...,coverage-first minimum
3,greedy_tree_path,CA3 Granule,10,11,2369.691494,2369.691494,SUB CA1 Projecting Pyramidal,CA3 Granule -> CA3 Pyramidal -> CA1 Radiatum G...,coverage-first minimum
4,greedy_tree_path,CA3 Pyramidal,9,10,2368.749475,2368.749475,SUB CA1 Projecting Pyramidal,CA3 Pyramidal -> CA1 Radiatum Giant -> CA1 Pyr...,coverage-first minimum


## Minimum Spanning Tree

The original notebook uses a maximum spanning tree as a high-weight undirected skeleton. This minimum-weight counterpart uses the weakest positive direction available for each unordered pair, builds a minimum spanning tree, then reports the lowest-weight valid directed tree path from each seed under the same coverage-first rule.

In [5]:
def minimum_spanning_tree_edges(transition_weights, eps=EPS):
    n = transition_weights.shape[0]
    sym_edges = []
    for i in range(n):
        for j in range(i + 1, n):
            candidates = []
            if transition_weights[i, j] > eps:
                candidates.append(float(transition_weights[i, j]))
            if transition_weights[j, i] > eps:
                candidates.append(float(transition_weights[j, i]))
            if candidates:
                sym_edges.append((min(candidates), i, j))
    sym_edges.sort(key=lambda item: item[0])

    dsu = DSU(n)
    tree_edges = []
    for weight, i, j in sym_edges:
        if dsu.union(i, j):
            tree_edges.append((i, j, weight))
            if len(tree_edges) == n - 1:
                break
    return tree_edges


def best_min_tree_path_from_seed(seed, tree_adj, data):
    best_path = [seed]
    best_score = _path_score(best_path)
    stack = [(seed, -1, [seed])]

    while stack:
        node, parent, path = stack.pop()
        children = [child for child, _ in tree_adj[node] if child != parent]
        if not children and _path_is_valid(path):
            score = _path_score(path)
            if _coverage_min_better(path, score, best_path, best_score):
                best_path, best_score = path, score
        for child in children:
            if data.transition_weights[node, child] <= EPS:
                continue
            stack.append((child, node, path + [child]))
    return best_path


min_tree_edges = minimum_spanning_tree_edges(data.transition_weights)
tree_adj = build_tree_adjacency(len(data.excitatory_nodes), min_tree_edges)

mst_results = []
mst_path_edges = []
for seed in range(len(data.excitatory_nodes)):
    path = best_min_tree_path_from_seed(seed, tree_adj, data)
    method = "minimum_spanning_tree"
    mst_results.append(_method_record(seed, method, path, {"objective_direction": "coverage-first minimum"}))
    mst_path_edges.extend(_method_edges(seed, method, path))

mst_df = pd.DataFrame(mst_results)
mst_df.head()


,method,seed,length_edges,n_nodes_visited,transition_weight_sum,summed_weight,terminus,path,objective_direction
0,minimum_spanning_tree,CA1 Pyramidal,0,1,0.000000,0.000000,CA1 Pyramidal,CA1 Pyramidal,coverage-first minimum
1,minimum_spanning_tree,CA1 Radiatum Giant,1,2,52.526068,52.526068,CA1 Pyramidal,CA1 Radiatum Giant -> CA1 Pyramidal,coverage-first minimum
2,minimum_spanning_tree,CA2 Pyramidal,2,3,315.590575,315.590575,CA1 Pyramidal,CA2 Pyramidal -> CA1 Radiatum Giant -> CA1 Pyr...,coverage-first minimum
3,minimum_spanning_tree,CA3 Granule,0,1,0.000000,0.000000,CA3 Granule,CA3 Granule,coverage-first minimum
4,minimum_spanning_tree,CA3 Pyramidal,1,2,3063.703303,3063.703303,CA3 Granule,CA3 Pyramidal -> CA3 Granule,coverage-first minimum


## Branch-and-Bound

The branch-and-bound adaptation searches simple directed paths from each seed. It prefers greater coverage first and lower weight second, with the same time and state-count guardrails as the maximum-weight notebook.

In [6]:
def minimum_branch_bound_path(seed, data, time_limit=0.25, node_limit=50_000):
    start_time = time.time()
    best_path = [seed]
    best_score = _path_score(best_path)
    states_seen = 0
    stopped_by = "complete"

    def dfs(node, visited, path, score):
        nonlocal best_path, best_score, states_seen, stopped_by
        states_seen += 1
        if states_seen >= node_limit:
            stopped_by = "node_limit"
            return
        if time.time() - start_time >= time_limit:
            stopped_by = "time_limit"
            return

        if _coverage_min_better(path, score, best_path, best_score):
            best_path = path.copy()
            best_score = score

        candidates = [(v, w) for v, w in data.adjacency[node] if v not in visited]
        candidates.sort(key=lambda item: item[1])
        for nxt, weight in candidates:
            visited.add(nxt)
            path.append(nxt)
            dfs(nxt, visited, path, score + float(weight) + float(data.self_weights[nxt]))
            path.pop()
            visited.remove(nxt)
            if stopped_by != "complete":
                return

    dfs(seed, {seed}, [seed], float(data.self_weights[seed]))
    return best_path, best_score, states_seen, stopped_by


bb_results = []
bb_edges = []
for seed in range(len(data.excitatory_nodes)):
    path, score, states, status = minimum_branch_bound_path(
        seed,
        data,
        time_limit=BB_TIME_LIMIT_PER_SEED,
        node_limit=BB_NODE_LIMIT_PER_SEED,
    )
    method = "branch_and_bound"
    bb_results.append(_method_record(seed, method, path, {"bb_states_seen": states, "bb_status": status, "objective_direction": "coverage-first minimum"}))
    bb_edges.extend(_method_edges(seed, method, path))

bb_df = pd.DataFrame(bb_results)
bb_df.head()


,method,seed,length_edges,n_nodes_visited,transition_weight_sum,summed_weight,terminus,path,bb_states_seen,bb_status,objective_direction
0,branch_and_bound,CA1 Pyramidal,30,31,29723.367151,29723.367151,CA1 Radiatum Giant,CA1 Pyramidal -> MEC LV Superficial Pyramidal ...,50000,node_limit,coverage-first minimum
1,branch_and_bound,CA1 Radiatum Giant,30,31,29767.605494,29767.605494,SUB CA1 Projecting Pyramidal,CA1 Radiatum Giant -> CA1 Pyramidal -> MEC LV ...,50000,node_limit,coverage-first minimum
2,branch_and_bound,CA2 Pyramidal,30,31,25716.342508,25716.342508,SUB CA1 Projecting Pyramidal,CA2 Pyramidal -> CA1 Radiatum Giant -> CA1 Pyr...,50000,node_limit,coverage-first minimum
3,branch_and_bound,CA3 Granule,30,31,27291.590146,27291.590146,SUB CA1 Projecting Pyramidal,CA3 Granule -> CA3 Pyramidal -> CA1 Radiatum G...,50000,node_limit,coverage-first minimum
4,branch_and_bound,CA3 Pyramidal,30,31,26412.092901,26412.092901,SUB CA1 Projecting Pyramidal,CA3 Pyramidal -> CA1 Radiatum Giant -> CA1 Pyr...,50000,node_limit,coverage-first minimum


## Dynamic Programming

The exact bitmask version is used only when the node count is small enough. For the 32-node EE graph, the notebook uses a beam-limited dynamic program that keeps high-coverage, low-weight states.

In [7]:
def exact_min_coverage_dp_path(seed, data):
    n = len(data.excitatory_nodes)
    seed_mask = 1 << seed
    dp = {(seed_mask, seed): float(data.self_weights[seed])}
    parent = {}
    best_key = (seed_mask, seed)
    best_score = float(data.self_weights[seed])

    for size in range(1, n + 1):
        current_items = [(key, value) for key, value in dp.items() if key[0].bit_count() == size]
        for (mask, last), score in current_items:
            if mask.bit_count() > best_key[0].bit_count() or (
                mask.bit_count() == best_key[0].bit_count() and score < best_score - EPS
            ):
                best_key = (mask, last)
                best_score = score
            for nxt, weight in data.adjacency[last]:
                if mask & (1 << nxt):
                    continue
                new_mask = mask | (1 << nxt)
                new_key = (new_mask, nxt)
                new_score = score + float(weight) + float(data.self_weights[nxt])
                if new_score < dp.get(new_key, math.inf) - EPS:
                    dp[new_key] = new_score
                    parent[new_key] = (mask, last)

    key = best_key
    path = [key[1]]
    while key in parent:
        key = parent[key]
        path.append(key[1])
    path.reverse()
    return path, best_score, "exact_min_coverage"


def beam_min_coverage_dp_path(seed, data, beam_width=5_000):
    n = len(data.excitatory_nodes)
    states = {(1 << seed, seed): (float(data.self_weights[seed]), [seed])}
    best_path = [seed]
    best_score = float(data.self_weights[seed])

    for _ in range(n - 1):
        new_states = dict(states)
        for (mask, last), (score, path) in states.items():
            if _coverage_min_better(path, score, best_path, best_score):
                best_path, best_score = path, score
            for nxt, weight in data.adjacency[last]:
                if mask & (1 << nxt):
                    continue
                new_mask = mask | (1 << nxt)
                new_key = (new_mask, nxt)
                new_score = score + float(weight) + float(data.self_weights[nxt])
                old = new_states.get(new_key)
                if old is None or new_score < old[0] - EPS:
                    new_states[new_key] = (new_score, path + [nxt])
        if len(new_states) > beam_width:
            ranked = sorted(new_states.items(), key=lambda item: (-len(item[1][1]), item[1][0]))[:beam_width]
            new_states = dict(ranked)
        if len(new_states) == len(states):
            break
        states = new_states

    for _, (score, path) in states.items():
        if _coverage_min_better(path, score, best_path, best_score):
            best_path, best_score = path, score
    return best_path, best_score, f"beam_width_{beam_width}_min_coverage"


dp_results = []
dp_edges = []
for seed in range(len(data.excitatory_nodes)):
    if len(data.excitatory_nodes) <= NOTEBOOK_EXACT_DP_MAX_E_NODES:
        path, score, mode = exact_min_coverage_dp_path(seed, data)
    else:
        path, score, mode = beam_min_coverage_dp_path(seed, data, beam_width=BEAM_WIDTH)
    method = "dynamic_programming"
    dp_results.append(_method_record(seed, method, path, {"dp_mode": mode, "objective_direction": "coverage-first minimum"}))
    dp_edges.extend(_method_edges(seed, method, path))

dp_df = pd.DataFrame(dp_results)
dp_df.head()


,method,seed,length_edges,n_nodes_visited,transition_weight_sum,summed_weight,terminus,path,dp_mode,objective_direction
0,dynamic_programming,CA1 Pyramidal,4,5,1376.162853,1376.162853,EC LI II Pyramidal Fan,CA1 Pyramidal -> MEC LV Superficial Pyramidal ...,beam_width_5000_min_coverage,coverage-first minimum
1,dynamic_programming,CA1 Radiatum Giant,5,6,1428.688921,1428.688921,EC LI II Pyramidal Fan,CA1 Radiatum Giant -> CA1 Pyramidal -> MEC LV ...,beam_width_5000_min_coverage,coverage-first minimum
2,dynamic_programming,CA2 Pyramidal,5,6,1313.432266,1313.432266,CA1 Radiatum Giant,CA2 Pyramidal -> CA3 Granule -> CA3c Pyramidal...,beam_width_5000_min_coverage,coverage-first minimum
3,dynamic_programming,CA3 Granule,6,7,1431.846426,1431.846426,CA1 Pyramidal,CA3 Granule -> CA3c Pyramidal -> DG Semilunar ...,beam_width_5000_min_coverage,coverage-first minimum
4,dynamic_programming,CA3 Pyramidal,5,6,1718.665986,1718.665986,MEC LV Pyramidal,CA3 Pyramidal -> CA1 Radiatum Giant -> CA1 Pyr...,beam_width_5000_min_coverage,coverage-first minimum


## Beam Path Plus Local Insertion

This minimum adaptation starts with the beam-limited coverage-first dynamic program, then tries local node relocations that keep the path valid and reduce total weight without changing coverage.

In [8]:
def _local_relocate_min_path(path, fixed_start=True, max_passes=25):
    path = list(path)
    best_score = _path_score(path)
    min_insert_pos = 1 if fixed_start else 0

    for _ in range(max_passes):
        best_candidate = None
        best_candidate_score = best_score
        for remove_pos in range(len(path)):
            if fixed_start and remove_pos == 0:
                continue
            node = path[remove_pos]
            remainder = path[:remove_pos] + path[remove_pos + 1:]
            for insert_pos in range(min_insert_pos, len(remainder) + 1):
                if insert_pos == remove_pos or insert_pos == remove_pos + 1:
                    continue
                candidate = remainder[:insert_pos] + [node] + remainder[insert_pos:]
                if not _path_is_valid(candidate):
                    continue
                candidate_score = _path_score(candidate)
                if candidate_score < best_candidate_score - EPS:
                    best_candidate = candidate
                    best_candidate_score = candidate_score
        if best_candidate is None:
            break
        path = best_candidate
        best_score = best_candidate_score
    return path, best_score


def minimum_beam_path_plus_local(seed, data, beam_width=5_000, max_local_passes=25):
    path, score, mode = beam_min_coverage_dp_path(seed, data, beam_width=beam_width)
    improved_path, improved_score = _local_relocate_min_path(path, fixed_start=True, max_passes=max_local_passes)
    return improved_path, improved_score, f"{mode}_plus_local_relocate"


beam_local_results = []
beam_local_edges = []
for seed in range(len(data.excitatory_nodes)):
    path, score, mode = minimum_beam_path_plus_local(seed, data, beam_width=BEAM_WIDTH)
    method = "beam_path_plus_local"
    beam_local_results.append(_method_record(seed, method, path, {"beam_local_mode": mode, "objective_direction": "coverage-first minimum"}))
    beam_local_edges.extend(_method_edges(seed, method, path))

beam_local_df = pd.DataFrame(beam_local_results)
beam_local_df.head()


,method,seed,length_edges,n_nodes_visited,transition_weight_sum,summed_weight,terminus,path,beam_local_mode,objective_direction
0,beam_path_plus_local,CA1 Pyramidal,4,5,1376.162853,1376.162853,EC LI II Pyramidal Fan,CA1 Pyramidal -> MEC LV Superficial Pyramidal ...,beam_width_5000_min_coverage_plus_local_relocate,coverage-first minimum
1,beam_path_plus_local,CA1 Radiatum Giant,5,6,1428.688921,1428.688921,EC LI II Pyramidal Fan,CA1 Radiatum Giant -> CA1 Pyramidal -> MEC LV ...,beam_width_5000_min_coverage_plus_local_relocate,coverage-first minimum
2,beam_path_plus_local,CA2 Pyramidal,5,6,1313.432266,1313.432266,CA1 Radiatum Giant,CA2 Pyramidal -> CA3 Granule -> CA3c Pyramidal...,beam_width_5000_min_coverage_plus_local_relocate,coverage-first minimum
3,beam_path_plus_local,CA3 Granule,6,7,1431.846426,1431.846426,CA1 Pyramidal,CA3 Granule -> CA3c Pyramidal -> DG Semilunar ...,beam_width_5000_min_coverage_plus_local_relocate,coverage-first minimum
4,beam_path_plus_local,CA3 Pyramidal,5,6,1718.665986,1718.665986,MEC LV Pyramidal,CA3 Pyramidal -> CA1 Radiatum Giant -> CA1 Pyr...,beam_width_5000_min_coverage_plus_local_relocate,coverage-first minimum


## MILP with Subtour Elimination

The MILP adaptation also uses coverage-first minimum scoring. It maximizes a large reward for each selected node and subtracts selected edge weight, so adding a reachable node is always preferred before minimizing path weight. This section is skipped when PuLP is not installed.

In [9]:
def minimum_milp_subtour_elimination_path(seed, data, time_limit=30.0, msg=False):
    try:
        import pulp
    except ImportError as exc:
        raise ImportError("PuLP is required for MILP minimum path search. Install it with `python -m pip install pulp`.") from exc

    n = len(data.excitatory_nodes)
    edges = [
        (i, j, float(data.transition_weights[i, j]))
        for i in range(n)
        for j in range(n)
        if i != j and data.transition_weights[i, j] > EPS
    ]
    incoming = {i: [] for i in range(n)}
    outgoing = {i: [] for i in range(n)}
    for i, j, _ in edges:
        outgoing[i].append((i, j))
        incoming[j].append((i, j))

    big_reward = float(data.transition_weights.sum() + data.self_weights.sum() + 1.0)
    problem = pulp.LpProblem("min_coverage_first_simple_directed_path", pulp.LpMaximize)
    x = {(i, j): pulp.LpVariable(f"x_{i}_{j}", lowBound=0, upBound=1, cat="Binary") for i, j, _ in edges}
    y = {i: pulp.LpVariable(f"y_{i}", lowBound=0, upBound=1, cat="Binary") for i in range(n)}
    order = {i: pulp.LpVariable(f"u_{i}", lowBound=0, upBound=n - 1, cat="Continuous") for i in range(n)}

    problem += (
        big_reward * pulp.lpSum(y[i] for i in range(n))
        - pulp.lpSum(w * x[(i, j)] for i, j, w in edges)
        - pulp.lpSum(float(data.self_weights[i]) * y[i] for i in range(n))
    )
    problem += y[seed] == 1
    problem += pulp.lpSum(x[e] for e in incoming[seed]) == 0
    problem += order[seed] == 0

    for i in range(n):
        problem += pulp.lpSum(x[e] for e in outgoing[i]) <= y[i]
        if i == seed:
            continue
        problem += pulp.lpSum(x[e] for e in incoming[i]) == y[i]
        problem += order[i] >= y[i]
        problem += order[i] <= (n - 1) * y[i]

    big_m = n
    for i, j, _ in edges:
        problem += order[j] >= order[i] + 1 - big_m * (1 - x[(i, j)])

    solver = pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit)
    problem.solve(solver)
    solution_status = getattr(problem, "sol_status", None)
    if solution_status is None:
        status = pulp.LpStatus.get(problem.status, str(problem.status))
        proven_optimal = False
    else:
        status = pulp.LpSolution.get(solution_status, str(solution_status))
        proven_optimal = solution_status == pulp.LpSolutionOptimal

    selected_edges = {i: j for (i, j), var in x.items() if var.value() is not None and var.value() > 0.5}
    path = [seed]
    visited = {seed}
    while path[-1] in selected_edges:
        nxt = selected_edges[path[-1]]
        if nxt in visited:
            raise ValueError("MILP returned a cycle despite subtour constraints.")
        path.append(nxt)
        visited.add(nxt)

    objective = float(pulp.value(problem.objective) or 0.0)
    return path, _path_score(path), status, objective, proven_optimal


milp_results = []
milp_edges = []
try:
    for seed in range(len(data.excitatory_nodes)):
        path, score, status, objective, proven_optimal = minimum_milp_subtour_elimination_path(
            seed,
            data,
            time_limit=MILP_TIME_LIMIT_PER_SEED,
        )
        method = "milp_subtour_elimination"
        milp_results.append(_method_record(seed, method, path, {"milp_status": status, "milp_objective": objective, "milp_proven_optimal": proven_optimal, "objective_direction": "coverage-first minimum"}))
        milp_edges.extend(_method_edges(seed, method, path))
except ImportError as exc:
    print(exc)
    print("Skipping MILP rows; non-MILP minimum methods remain available below.")

milp_df = pd.DataFrame(milp_results)
milp_df.head() if not milp_df.empty else milp_df


PuLP is required for MILP minimum path search. Install it with `python -m pip install pulp`.
Skipping MILP rows; non-MILP minimum methods remain available below.


""


## Minimum-Weight Asymmetric Hamiltonian Path

This is the full-coverage counterpart to the original maximum-weight Hamiltonian path. Every seed is forced to start a path visiting all 32 excitatory nodes exactly once, and the objective minimizes summed transition weight. This section is skipped when PuLP is not installed.

In [10]:
def minimum_weight_asymmetric_hamiltonian_path(seed, data, time_limit=30.0, msg=False):
    try:
        import pulp
    except ImportError as exc:
        raise ImportError("PuLP is required for Hamiltonian path search. Install it with `python -m pip install pulp`.") from exc

    n = len(data.excitatory_nodes)
    edges = [
        (i, j, float(data.transition_weights[i, j]))
        for i in range(n)
        for j in range(n)
        if i != j and data.transition_weights[i, j] > EPS
    ]
    incoming = {i: [] for i in range(n)}
    outgoing = {i: [] for i in range(n)}
    for i, j, _ in edges:
        outgoing[i].append((i, j))
        incoming[j].append((i, j))

    problem = pulp.LpProblem("min_weight_asymmetric_hamiltonian_path", pulp.LpMinimize)
    x = {(i, j): pulp.LpVariable(f"x_{i}_{j}", lowBound=0, upBound=1, cat="Binary") for i, j, _ in edges}
    order = {i: pulp.LpVariable(f"u_{i}", lowBound=0, upBound=n - 1, cat="Continuous") for i in range(n)}

    problem += pulp.lpSum(w * x[(i, j)] for i, j, w in edges) + float(data.self_weights.sum())
    problem += pulp.lpSum(x[e] for e in incoming[seed]) == 0
    if n > 1:
        problem += pulp.lpSum(x[e] for e in outgoing[seed]) == 1
    problem += order[seed] == 0
    problem += pulp.lpSum(x[(i, j)] for i, j, _ in edges) == n - 1

    for i in range(n):
        problem += pulp.lpSum(x[e] for e in outgoing[i]) <= 1
        if i == seed:
            continue
        problem += pulp.lpSum(x[e] for e in incoming[i]) == 1
        problem += order[i] >= 1

    big_m = n
    for i, j, _ in edges:
        problem += order[j] >= order[i] + 1 - big_m * (1 - x[(i, j)])

    solver = pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit)
    problem.solve(solver)
    solution_status = getattr(problem, "sol_status", None)
    if solution_status is None:
        status = pulp.LpStatus.get(problem.status, str(problem.status))
        proven_optimal = False
    else:
        status = pulp.LpSolution.get(solution_status, str(solution_status))
        proven_optimal = solution_status == pulp.LpSolutionOptimal

    selected_edges = {i: j for (i, j), var in x.items() if var.value() is not None and var.value() > 0.5}
    path = [seed]
    visited = {seed}
    while path[-1] in selected_edges:
        nxt = selected_edges[path[-1]]
        if nxt in visited:
            raise ValueError("Hamiltonian MILP returned a cycle despite subtour constraints.")
        path.append(nxt)
        visited.add(nxt)
    if len(path) != n:
        raise ValueError(f"Expected a {n}-node Hamiltonian path, reconstructed {len(path)} nodes.")

    objective = float(pulp.value(problem.objective) or 0.0)
    return path, _path_score(path), status, objective, proven_optimal


hamiltonian_results = []
hamiltonian_edges = []
try:
    for seed in range(len(data.excitatory_nodes)):
        path, score, status, objective, proven_optimal = minimum_weight_asymmetric_hamiltonian_path(
            seed,
            data,
            time_limit=HAMILTONIAN_TIME_LIMIT_PER_SEED,
        )
        method = "minimum_weight_asymmetric_hamiltonian_path"
        hamiltonian_results.append(_method_record(seed, method, path, {"hamiltonian_status": status, "hamiltonian_objective": objective, "hamiltonian_proven_optimal": proven_optimal, "objective_direction": "minimum Hamiltonian"}))
        hamiltonian_edges.extend(_method_edges(seed, method, path))
except ImportError as exc:
    print(exc)
    print("Skipping exact Hamiltonian MILP rows; beam/local and other minimum methods remain available below.")

hamiltonian_df = pd.DataFrame(hamiltonian_results)
hamiltonian_df.head() if not hamiltonian_df.empty else hamiltonian_df


PuLP is required for Hamiltonian path search. Install it with `python -m pip install pulp`.
Skipping exact Hamiltonian MILP rows; beam/local and other minimum methods remain available below.


""


## Feedback-Arc Ordering

The maximum-weight notebook finds an ordering with large forward edge mass. A minimum-forward contrast is obtained by reversing that high-forward order, which makes the original graph's previously forward-heavy direction mostly backward. Per-seed paths are then computed as coverage-first minimum paths within the resulting forward-edge DAG.

In [11]:
def permutation_null_min_forward_fraction(weights, observed_fraction, n_null=1_000, random_seed=2026, eps=EPS):
    rng = np.random.default_rng(random_seed)
    n = weights.shape[0]
    fractions = np.empty(n_null)
    for draw in range(n_null):
        fractions[draw] = ordering_metrics(rng.permutation(n).tolist(), weights, eps=eps)["forward_weight_fraction"]
    n_at_most = int(np.sum(fractions <= observed_fraction))
    return {
        "null_n": n_null,
        "null_mean_forward_fraction": float(fractions.mean()),
        "null_sd_forward_fraction": float(fractions.std(ddof=1)),
        "null_min_forward_fraction": float(fractions.min()),
        "observed_percentile_in_null": float(100.0 * np.mean(fractions < observed_fraction)),
        "empirical_p_min_forward_fraction": (1 + n_at_most) / (1 + n_null),
    }


def shortest_forward_path_from_seed_coverage(seed, order, data, eps=EPS):
    weights = data.transition_weights
    n = weights.shape[0]
    rank = np.empty(n, dtype=int)
    for position, node in enumerate(order):
        rank[node] = position

    best_len = np.zeros(n, dtype=int)
    best_score = np.full(n, math.inf)
    predecessor = np.full(n, -1, dtype=int)
    best_len[seed] = 1
    best_score[seed] = float(data.self_weights[seed])

    for node in order:
        if best_len[node] == 0:
            continue
        for target, weight in data.adjacency[node]:
            if rank[target] <= rank[node]:
                continue
            candidate_len = best_len[node] + 1
            candidate_score = best_score[node] + float(weight) + float(data.self_weights[target])
            if candidate_len > best_len[target] or (
                candidate_len == best_len[target] and candidate_score < best_score[target] - eps
            ):
                best_len[target] = candidate_len
                best_score[target] = candidate_score
                predecessor[target] = node

    finite = np.flatnonzero(best_len > 0)
    terminal = min(finite.tolist(), key=lambda node: (-best_len[node], best_score[node], data.excitatory_nodes[node]))
    path = [terminal]
    while predecessor[path[-1]] != -1:
        path.append(int(predecessor[path[-1]]))
    path.reverse()
    return path, float(best_score[terminal])


def min_forward_ordering_path_records(order, data, method="feedback_arc_ordering"):
    records = []
    edge_rows = []
    metrics = ordering_metrics(order, data.transition_weights)
    for seed in range(len(data.excitatory_nodes)):
        path, score = shortest_forward_path_from_seed_coverage(seed, order, data)
        records.append(path_to_records(seed, method, path, data, {"ordering_dag_score": score, "ordering_exact_within_ordering": True, "ordering_forward_weight_fraction": metrics["forward_weight_fraction"], "seed_position": int(order.index(seed)), "objective_direction": "minimum-forward ordering, coverage-first minimum path"}))
        edge_rows.extend(edges_from_path(seed, method, path, data))
    return records, edge_rows


max_forward_ordering = feedback_arc_ordering(data, n_restarts=ORDERING_RESTARTS, random_seed=2026)
minimum_forward_order = list(reversed(max_forward_ordering.order))
ordering_metrics_min = ordering_metrics(minimum_forward_order, data.transition_weights)
ordering_null = permutation_null_min_forward_fraction(
    data.transition_weights,
    ordering_metrics_min["forward_weight_fraction"],
    n_null=ORDERING_NULL_DRAWS,
    random_seed=2026,
)

print(f"Strongly connected components in source high-forward run: {len(max_forward_ordering.scc_sizes)}")
print(f"Component sizes: {max_forward_ordering.scc_sizes}")
print(f"Minimum-forward fraction after reversing high-forward order: {ordering_metrics_min['forward_weight_fraction']:.4f}")
for key, value in ordering_null.items():
    print(f"  {key:34s} {value:.6g}")


class _MinimumOrderingResult:
    def __init__(self, order, labels, metrics):
        self.order = order
        self.labels = labels
        self.metrics = metrics

    @property
    def ordered_labels(self):
        return [self.labels[i] for i in self.order]

    def position_table(self):
        return pd.DataFrame({"position": np.arange(len(self.order)), "node": self.ordered_labels, "node_index": self.order})


ordering = _MinimumOrderingResult(minimum_forward_order, list(data.excitatory_nodes), ordering_metrics_min)
ordering_positions = ordering.position_table()
ordering_results, ordering_edges = min_forward_ordering_path_records(ordering.order, data)
ordering_df = pd.DataFrame(ordering_results)
ordering_all_edges = ordering_edge_table(ordering, data)

print("Heaviest forward transitions under the minimum-forward ordering:")
print(ordering_all_edges[ordering_all_edges["direction"].eq("forward")].sort_values("weight", ascending=False).head(10).to_string(index=False))
ordering_positions.head(12)


Strongly connected components in source high-forward run: 1
Component sizes: [32]
Minimum-forward fraction after reversing high-forward order: 0.0504
  null_n                             2000
  null_mean_forward_fraction         0.498314
  null_sd_forward_fraction           0.121932
  null_min_forward_fraction          0.15191
  observed_percentile_in_null        0
  empirical_p_min_forward_fraction   0.00049975
Heaviest forward transitions under the minimum-forward ordering:
                             sender                      receiver  sender_position  receiver_position  displacement      weight direction
      EC LII III Pyramidal Tripolar EC LI II Multipolar Pyramidal               29                 30             1 4216.395704   forward
                         DG Granule                 CA2 Pyramidal                3                 24            21 3360.733225   forward
      EC LI II Multipolar Pyramidal        EC LI II Pyramidal Fan               30                 31    

,position,node,node_index
0,0,CA3 Granule,3
1,1,DG Hilar Ectopic Granule,7
2,2,DG Mossy,8
3,3,DG Granule,6
4,4,MEC LV VI Pyramidal Polymorphic,30
5,5,SUB CA1 Projecting Pyramidal,31
6,6,EC LIII Pyramidal,13
7,7,EC LIII Stellate,15
8,8,MEC LII Oblique Pyramidal,23
9,9,MEC LIII Complex Pyramidal,26


## Combine method results

This section stacks the adapted methods into the same long-table shape as the original notebook.

In [12]:
comparison_parts = [greedy_df, mst_df, bb_df, dp_df, beam_local_df, milp_df, hamiltonian_df, ordering_df]
comparison = pd.concat([part for part in comparison_parts if not part.empty], ignore_index=True)
edge_table = pd.DataFrame(
    greedy_edges
    + mst_path_edges
    + bb_edges
    + dp_edges
    + beam_local_edges
    + milp_edges
    + hamiltonian_edges
    + ordering_edges
)

comparison.sort_values(["seed", "method"]).reset_index(drop=True).head(20)


,method,seed,length_edges,n_nodes_visited,transition_weight_sum,summed_weight,terminus,path,objective_direction,bb_states_seen,bb_status,dp_mode,beam_local_mode,ordering_dag_score,ordering_exact_within_ordering,ordering_forward_weight_fraction,seed_position
0,beam_path_plus_local,CA1 Pyramidal,4,5,1376.162853,1376.162853,EC LI II Pyramidal Fan,CA1 Pyramidal -> MEC LV Superficial Pyramidal ...,coverage-first minimum,NaN,NaN,NaN,beam_width_5000_min_coverage_plus_local_relocate,NaN,NaN,NaN,NaN
1,branch_and_bound,CA1 Pyramidal,30,31,29723.367151,29723.367151,CA1 Radiatum Giant,CA1 Pyramidal -> MEC LV Superficial Pyramidal ...,coverage-first minimum,50000.0,node_limit,NaN,NaN,NaN,NaN,NaN,NaN
2,dynamic_programming,CA1 Pyramidal,4,5,1376.162853,1376.162853,EC LI II Pyramidal Fan,CA1 Pyramidal -> MEC LV Superficial Pyramidal ...,coverage-first minimum,NaN,NaN,beam_width_5000_min_coverage,NaN,NaN,NaN,NaN,NaN
3,feedback_arc_ordering,CA1 Pyramidal,0,1,0.000000,0.000000,CA1 Pyramidal,CA1 Pyramidal,"minimum-forward ordering, coverage-first minim...",NaN,NaN,NaN,NaN,0.000000,True,0.050396,22.0
4,greedy_tree_path,CA1 Pyramidal,8,9,2020.848183,2020.848183,CA1 Radiatum Giant,CA1 Pyramidal -> MEC LV Superficial Pyramidal ...,coverage-first minimum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,minimum_spanning_tree,CA1 Pyramidal,0,1,0.000000,0.000000,CA1 Pyramidal,CA1 Pyramidal,coverage-first minimum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,beam_path_plus_local,CA1 Radiatum Giant,5,6,1428.688921,1428.688921,EC LI II Pyramidal Fan,CA1 Radiatum Giant -> CA1 Pyramidal -> MEC LV ...,coverage-first minimum,NaN,NaN,NaN,beam_width_5000_min_coverage_plus_local_relocate,NaN,NaN,NaN,NaN
7,branch_and_bound,CA1 Radiatum Giant,30,31,29767.605494,29767.605494,SUB CA1 Projecting Pyramidal,CA1 Radiatum Giant -> CA1 Pyramidal -> MEC LV ...,coverage-first minimum,50000.0,node_limit,NaN,NaN,NaN,NaN,NaN,NaN
8,dynamic_programming,CA1 Radiatum Giant,5,6,1428.688921,1428.688921,EC LI II Pyramidal Fan,CA1 Radiatum Giant -> CA1 Pyramidal -> MEC LV ...,coverage-first minimum,NaN,NaN,beam_width_5000_min_coverage,NaN,NaN,NaN,NaN,NaN
9,feedback_arc_ordering,CA1 Radiatum Giant,0,1,0.000000,0.000000,CA1 Radiatum Giant,CA1 Radiatum Giant,"minimum-forward ordering, coverage-first minim...",NaN,NaN,NaN,NaN,0.000000,True,0.050396,23.0


## Write outputs

The output files mirror the original notebook but are written to `ee_backbone_min_weight_comparison_outputs`. The method summary is self-describing because these are minimum-weight adaptations, not the original maximum-weight objectives.

In [13]:
def build_wide_comparison(comparison):
    values = ["length_edges", "n_nodes_visited", "transition_weight_sum", "summed_weight", "terminus"]
    wide = comparison.pivot_table(index="seed", columns="method", values=values, aggfunc="first")
    wide.columns = [f"{metric}__{method}" for metric, method in wide.columns]
    return wide.reset_index()


def summarize_min_methods(comparison):
    summary = (
        comparison.groupby("method", dropna=False)
        .agg(
            n_seeds=("seed", "nunique"),
            mean_length=("length_edges", "mean"),
            median_length=("length_edges", "median"),
            min_summed_weight=("summed_weight", "min"),
            mean_summed_weight=("summed_weight", "mean"),
            median_summed_weight=("summed_weight", "median"),
            max_summed_weight=("summed_weight", "max"),
            n_unique_termini=("terminus", "nunique"),
        )
        .reset_index()
    )
    objective = {
        "greedy_tree_path": "coverage-first path using weakest local outgoing transition",
        "minimum_spanning_tree": "low-weight undirected skeleton, then coverage-first directed tree path",
        "branch_and_bound": "coverage-first minimum simple path, limited by time/node guardrails",
        "dynamic_programming": "coverage-first minimum simple path, exact only when node count is small enough",
        "beam_path_plus_local": "beam-limited coverage-first minimum path plus local relocation",
        "milp_subtour_elimination": "coverage-first minimum simple path MILP",
        "minimum_weight_asymmetric_hamiltonian_path": "minimum-weight path visiting every node once",
        "feedback_arc_ordering": "minimum-forward global ordering; per-seed path is coverage-first minimum within that DAG",
    }
    summary.insert(1, "objective", summary["method"].map(objective).fillna(""))
    summary.insert(2, "comparable_on_summed_weight", summary["method"].ne("feedback_arc_ordering"))
    return summary


def build_min_milp_benchmark_table(comparison):
    if "milp_subtour_elimination" not in set(comparison["method"]):
        return pd.DataFrame()
    milp = comparison[comparison["method"].eq("milp_subtour_elimination")].set_index("seed")
    rows = []
    for _, row in comparison.iterrows():
        if row["method"] == "milp_subtour_elimination" or row["seed"] not in milp.index:
            continue
        milp_row = milp.loc[row["seed"]]
        rows.append(
            {
                "seed": row["seed"],
                "method": row["method"],
                "method_nodes": row["n_nodes_visited"],
                "milp_nodes": milp_row["n_nodes_visited"],
                "method_summed_weight": row["summed_weight"],
                "milp_summed_weight": milp_row["summed_weight"],
                "weight_minus_milp": row["summed_weight"] - milp_row["summed_weight"],
                "milp_proven_optimal": bool(milp_row.get("milp_proven_optimal", False)),
                "same_or_more_coverage_than_milp": int(row["n_nodes_visited"]) >= int(milp_row["n_nodes_visited"]),
                "method_terminus": row["terminus"],
                "milp_terminus": milp_row["terminus"],
            }
        )
    return pd.DataFrame(rows).sort_values(["seed", "method"]).reset_index(drop=True)


OUTDIR.mkdir(exist_ok=True)
wide = build_wide_comparison(comparison)
method_summary = summarize_min_methods(comparison)
min_milp_benchmark = build_min_milp_benchmark_table(comparison)

comparison.to_csv(OUTDIR / "method_comparison_long.csv", index=False)
wide.to_csv(OUTDIR / "method_comparison_wide.csv", index=False)
edge_table.to_csv(OUTDIR / "all_methods_edges.csv", index=False)
method_summary.to_csv(OUTDIR / "method_summary.csv", index=False)
if not min_milp_benchmark.empty:
    min_milp_benchmark.to_csv(OUTDIR / "milp_benchmark_gaps.csv", index=False)
ordering_positions.to_csv(OUTDIR / "backbone_ordering.csv", index=False)
ordering_all_edges.to_csv(OUTDIR / "ordering_edge_directions.csv", index=False)
pd.Series(ordering.metrics).to_csv(OUTDIR / "ordering_metrics.csv")
pd.Series(ordering_null).to_csv(OUTDIR / "ordering_permutation_null.csv")

print("Wrote:")
for name in [
    "method_comparison_long.csv",
    "method_comparison_wide.csv",
    "all_methods_edges.csv",
    "method_summary.csv",
    "milp_benchmark_gaps.csv",
    "backbone_ordering.csv",
    "ordering_edge_directions.csv",
    "ordering_metrics.csv",
    "ordering_permutation_null.csv",
]:
    output_path = OUTDIR / name
    if output_path.exists():
        print(f"  {output_path}")

method_summary


Wrote:
  ee_backbone_min_weight_comparison_outputs\method_comparison_long.csv
  ee_backbone_min_weight_comparison_outputs\method_comparison_wide.csv
  ee_backbone_min_weight_comparison_outputs\all_methods_edges.csv
  ee_backbone_min_weight_comparison_outputs\method_summary.csv
  ee_backbone_min_weight_comparison_outputs\backbone_ordering.csv
  ee_backbone_min_weight_comparison_outputs\ordering_edge_directions.csv
  ee_backbone_min_weight_comparison_outputs\ordering_metrics.csv
  ee_backbone_min_weight_comparison_outputs\ordering_permutation_null.csv


,method,objective,comparable_on_summed_weight,n_seeds,mean_length,median_length,min_summed_weight,mean_summed_weight,median_summed_weight,max_summed_weight,n_unique_termini
0,beam_path_plus_local,beam-limited coverage-first minimum path plus ...,True,32,4.46875,4.0,43.944350,651.035859,354.662295,1910.352984,6
1,branch_and_bound,"coverage-first minimum simple path, limited by...",True,32,29.93750,30.0,13555.897045,25990.357068,27005.317831,31017.524574,4
2,dynamic_programming,"coverage-first minimum simple path, exact only...",True,32,4.46875,4.0,43.944350,651.035859,354.662295,1910.352984,6
3,feedback_arc_ordering,minimum-forward global ordering; per-seed path...,False,32,5.00000,4.5,0.000000,6116.163334,7924.223469,10338.545393,4
4,greedy_tree_path,coverage-first path using weakest local outgoi...,True,32,16.68750,19.5,2020.848183,10592.479482,13909.109449,22308.119842,5
5,minimum_spanning_tree,"low-weight undirected skeleton, then coverage-...",True,32,2.28125,2.5,0.000000,2304.116065,931.919563,10402.073341,9


## MILP benchmark gaps

When the optional MILP rows exist, this table compares each method against the coverage-first MILP result for the same seed. It is skipped automatically when PuLP is unavailable.

In [14]:
if min_milp_benchmark.empty:
    print("MILP benchmark table is empty because MILP rows were not generated in this environment.")
else:
    min_milp_benchmark.sort_values(["same_or_more_coverage_than_milp", "weight_minus_milp"], ascending=[False, True]).head(20)


MILP benchmark table is empty because MILP rows were not generated in this environment.


## Path weights by method

This plot compares the distribution of `summed_weight`. Because minimum methods are coverage-first, read this together with path length: lower weight is only meaningful among paths with similar coverage.

In [15]:
if plt is None:
    print("matplotlib is not installed; skipping boxplot.")
else:
    plot_df = comparison.loc[comparison["method"] != "feedback_arc_ordering", ["method", "summed_weight"]].copy()
    ax = plot_df.boxplot(column="summed_weight", by="method", rot=45)
    plt.suptitle("")
    plt.title("Minimum-adapted path weights by method")
    plt.xlabel("Method")
    plt.ylabel("Path weight")
    plt.tight_layout()
    plt.show()


matplotlib is not installed; skipping boxplot.


## Path comparison tables

These final tables follow the printout style at the bottom of `ee_backbone_method_comparison.ipynb`. The first table selects the highest-coverage path obtained by each method, breaking ties by lower `summed_weight`. The second fixes the seed to the canonical EC start.

In [16]:
PATH_METHOD_ORDER = [
    "greedy_tree_path",
    "minimum_spanning_tree",
    "branch_and_bound",
    "dynamic_programming",
    "beam_path_plus_local",
    "milp_subtour_elimination",
    "minimum_weight_asymmetric_hamiltonian_path",
    "feedback_arc_ordering",
]

PATH_METHOD_LABELS = {
    "greedy_tree_path": "Greedy tree/path",
    "minimum_spanning_tree": "Minimum spanning tree",
    "branch_and_bound": "Branch-and-bound",
    "dynamic_programming": "Dynamic programming / beam DP",
    "beam_path_plus_local": "Beam path plus local",
    "milp_subtour_elimination": "MILP subtour elimination",
    "minimum_weight_asymmetric_hamiltonian_path": "Minimum-weight Hamiltonian path",
    "feedback_arc_ordering": "Feedback-arc ordering",
}

PATH_COMPARISON_METRICS = ["seed", "length_edges", "n_nodes_visited", "summed_weight", "terminus", "path"]


def _method_column_table(rows):
    if rows.empty:
        return pd.DataFrame()
    available_methods = [method for method in PATH_METHOD_ORDER if method in set(rows["method"])]
    extra_methods = [method for method in rows["method"].drop_duplicates() if method not in available_methods]
    method_order = available_methods + extra_methods
    table = rows.set_index("method").reindex(method_order)[PATH_COMPARISON_METRICS].T
    table.columns = [PATH_METHOD_LABELS.get(method, method) for method in table.columns]
    if "summed_weight" in table.index:
        table.loc["summed_weight"] = table.loc["summed_weight"].map(lambda value: f"{float(value):,.2f}" if pd.notna(value) else value)
    return table


def _display_path_table(title, table):
    print(title)
    if table.empty:
        print("No rows available.")
    else:
        with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
            display(table)


best_coverage_paths_by_method = (
    comparison.sort_values(["method", "n_nodes_visited", "summed_weight"], ascending=[True, False, True])
    .groupby("method", sort=False)
    .head(1)
)
best_coverage_path_table = _method_column_table(best_coverage_paths_by_method)

canonical_ec_paths = comparison[comparison["seed"].eq(CANONICAL_EC_START)].copy()
if canonical_ec_paths.empty:
    ec_candidates = sorted(comparison.loc[comparison["seed"].str.startswith("EC "), "seed"].unique())
    raise ValueError(f"Canonical EC start {CANONICAL_EC_START!r} was not found. Available EC starts include: {ec_candidates[:8]}")
canonical_ec_path_table = _method_column_table(canonical_ec_paths)

_display_path_table("Highest-coverage minimum path obtained by each method", best_coverage_path_table)
_display_path_table(f"Minimum-adapted paths from canonical EC start: {CANONICAL_EC_START}", canonical_ec_path_table)


Highest-coverage minimum path obtained by each method
Minimum-adapted paths from canonical EC start: EC LI II Pyramidal Fan


,Greedy tree/path,Minimum spanning tree,Branch-and-bound,Dynamic programming / beam DP,Beam path plus local,Feedback-arc ordering
seed,LEC LVI Multipolar Pyramidal,LEC LVI Multipolar Pyramidal,MEC LV Pyramidal,DG Hilar Ectopic Granule,DG Hilar Ectopic Granule,EC LIII Pyramidal
length_edges,24,5,31,6,6,13
n_nodes_visited,25,6,32,7,7,14
summed_weight,"16,302.19","3,829.79","28,777.12",930.85,930.85,"10,065.04"
terminus,DG Hilar Ectopic Granule,MEC LIII Complex Pyramidal,SUB CA1 Projecting Pyramidal,CA1 Radiatum Giant,CA1 Radiatum Giant,EC LI II Pyramidal Fan
path,LEC LVI Multipolar Pyramidal -> EC LIII Stellate -> MEC LV Superficial Pyramidal -> EC LI II Pyramidal Fan -> EC LV Deep Pyramidal -> MEC LV Multipolar Pyramidal -> EC LIII Pyramidal -> MEC LV Pyramidal -> SUB CA1 Projecting Pyramidal -> CA1 Radiatum Giant -> CA1 Pyramidal -> LEC LIII Multipolar Principal -> EC LIV VI Deep Multipolar Principal -> EC LIII Pyramidal Stellate -> EC LIII V Bipolar Pyramidal -> MEC LII III Pyramidal Multiform -> EC LII III Pyramidal Tripolar -> DG Semilunar Granule -> CA3 Pyramidal -> CA3 Granule -> CA3c Pyramidal -> DG Mossy -> DG Granule -> CA2 Pyramidal -> DG Hilar Ectopic Granule,LEC LVI Multipolar Pyramidal -> LEC LIII Multipolar Principal -> EC LI II Pyramidal Fan -> MEC LIII Bipolar Complex Pyramidal -> EC LIV VI Deep Multipolar Principal -> MEC LIII Complex Pyramidal,MEC LV Pyramidal -> EC LI II Pyramidal Fan -> MEC LV Superficial Pyramidal -> EC LIV VI Deep Multipolar Principal -> EC LIII Pyramidal Stellate -> EC LIII V Bipolar Pyramidal -> MEC LII III Pyramidal Multiform -> EC LIII Pyramidal -> EC LIII Stellate -> LEC LIII Complex Pyramidal -> LEC LVI Multipolar Pyramidal -> EC LII III Pyramidal Tripolar -> DG Semilunar Granule -> CA3 Pyramidal -> CA1 Radiatum Giant -> CA1 Pyramidal -> MEC LV Multipolar Pyramidal -> MEC LV VI Pyramidal Polymorphic -> EC LV Deep Pyramidal -> LEC LIII Multipolar Principal -> EC LI II Multipolar Pyramidal -> MEC LIII Complex Pyramidal -> MEC LII Oblique Pyramidal -> MEC LIII Bipolar Complex Pyramidal -> MEC LII Stellate -> DG Hilar Ectopic Granule -> DG Mossy -> DG Granule -> CA2 Pyramidal -> CA3 Granule -> CA3c Pyramidal -> SUB CA1 Projecting Pyramidal,DG Hilar Ectopic Granule -> DG Mossy -> DG Semilunar Granule -> CA3c Pyramidal -> CA3 Granule -> CA3 Pyramidal -> CA1 Radiatum Giant,DG Hilar Ectopic Granule -> DG Mossy -> DG Semilunar Granule -> CA3c Pyramidal -> CA3 Granule -> CA3 Pyramidal -> CA1 Radiatum Giant,EC LIII Pyramidal -> EC LIII Stellate -> MEC LIII Complex Pyramidal -> MEC LIII Bipolar Complex Pyramidal -> MEC LV Multipolar Pyramidal -> MEC LV Superficial Pyramidal -> MEC LV Pyramidal -> EC LV Deep Pyramidal -> EC LIII V Bipolar Pyramidal -> EC LIV VI Deep Multipolar Principal -> EC LIII Pyramidal Stellate -> EC LII III Pyramidal Tripolar -> EC LI II Multipolar Pyramidal -> EC LI II Pyramidal Fan


,Greedy tree/path,Minimum spanning tree,Branch-and-bound,Dynamic programming / beam DP,Beam path plus local,Feedback-arc ordering
seed,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan
length_edges,21,3,30,4,4,0
n_nodes_visited,22,4,31,5,5,1
summed_weight,"15,425.70","3,493.88","25,493.65",455.79,455.79,0.00
terminus,DG Hilar Ectopic Granule,MEC LIII Complex Pyramidal,CA3 Pyramidal,EC LIII V Bipolar Pyramidal,EC LIII V Bipolar Pyramidal,EC LI II Pyramidal Fan
path,EC LI II Pyramidal Fan -> MEC LV Superficial Pyramidal -> EC LIV VI Deep Multipolar Principal -> MEC LV Pyramidal -> SUB CA1 Projecting Pyramidal -> CA1 Radiatum Giant -> CA1 Pyramidal -> LEC LIII Multipolar Principal -> LEC LVI Multipolar Pyramidal -> EC LIII Stellate -> EC LIII V Bipolar Pyramidal -> MEC LII III Pyramidal Multiform -> EC LIII Pyramidal -> EC LII III Pyramidal Tripolar -> DG Semilunar Granule -> CA3 Pyramidal -> CA3 Granule -> CA3c Pyramidal -> DG Mossy -> DG Granule -> CA2 Pyramidal -> DG Hilar Ectopic Granule,EC LI II Pyramidal Fan -> MEC LIII Bipolar Complex Pyramidal -> EC LIV VI Deep Multipolar Principal -> MEC LIII Complex Pyramidal,EC LI II Pyramidal Fan -> MEC LV Superficial Pyramidal -> EC LIV VI Deep Multipolar Principal -> MEC LV Pyramidal -> SUB CA1 Projecting Pyramidal -> CA1 Radiatum Giant -> CA1 Pyramidal -> LEC LIII Multipolar Principal -> LEC LVI Multipolar Pyramidal -> EC LIII Stellate -> EC LIII V Bipolar Pyramidal -> MEC LII III Pyramidal Multiform -> EC LIII Pyramidal -> EC LII III Pyramidal Tripolar -> EC LV Deep Pyramidal -> MEC LV Multipolar Pyramidal -> EC LIII Pyramidal Stellate -> LEC LIII Complex Pyramidal -> EC LI II Multipolar Pyramidal -> MEC LIII Complex Pyramidal -> MEC LII Oblique Pyramidal -> MEC LIII Bipolar Complex Pyramidal -> MEC LII Stellate -> DG Hilar Ectopic Granule -> DG Mossy -> DG Semilunar Granule -> DG Granule -> CA2 Pyramidal -> CA3 Granule -> CA3c Pyramidal -> CA3 Pyramidal,EC LI II Pyramidal Fan -> MEC LV Superficial Pyramidal -> EC LIII Pyramidal -> EC LIII Stellate -> EC LIII V Bipolar Pyramidal,EC LI II Pyramidal Fan -> MEC LV Superficial Pyramidal -> EC LIII Pyramidal -> EC LIII Stellate -> EC LIII V Bipolar Pyramidal,EC LI II Pyramidal Fan


## Walk a single seed/method path

This final cell keeps the same convenience pattern as the original notebook: choose one seed/method and print its path node by node with incoming edge weights and cumulative weight.

In [17]:
_ = describe_backbone_path(
    "CA1 Pyramidal",
    "greedy_tree_path",
    comparison,
    data,
    method_labels=PATH_METHOD_LABELS,
)


Greedy tree/path path from seed 'CA1 Pyramidal'
  9 node(s), 8 edge(s), total transition weight = 2,020.8482

   1. CA1 Pyramidal                  (start)
   2. MEC LV Superficial Pyramidal   +1316.2619   cumulative:  1316.2619
   3. EC LI II Pyramidal Fan         +  3.0378   cumulative:  1319.2997
   4. EC LV Deep Pyramidal           +426.2149   cumulative:  1745.5146
   5. MEC LV Multipolar Pyramidal    +169.0293   cumulative:  1914.5439
   6. EC LIII Pyramidal              + 26.5346   cumulative:  1941.0785
   7. MEC LV Pyramidal               + 12.7601   cumulative:  1953.8386
   8. SUB CA1 Projecting Pyramidal   + 58.7218   cumulative:  2012.5605
   9. CA1 Radiatum Giant             +  8.2877   cumulative:  2020.8482


## Takeaways

- Greedy path, spanning-tree path, branch-and-bound, dynamic programming, beam/local, MILP, Hamiltonian MILP, and feedback-arc ordering can all be adapted for minimum-weight analysis.
- Variable-length minimum paths need an anti-degeneracy rule; this notebook uses coverage first, then minimum weight.
- Feedback-arc ordering is a global ordering objective, not a simple path objective. Here it is used as a minimum-forward contrast by reversing the high-forward ordering and then finding coverage-first low-weight forward paths inside that ordering.
- Exact MILP sections require PuLP. If PuLP is unavailable, the notebook still produces the non-MILP minimum comparisons and leaves the MILP benchmark table empty.